In [ ]:

import numpy as np

from scripts.beamforming import get_best_beam
from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import RF_PARAM_5G, NETWORK_TYPE

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt("../data/random_seeds.csv", dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ["toa_pps", "toa_cir", "toa_cov", "campaign_id"]
df["measurements_matrix"] = df["measurements_matrix"].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

operator_choice = [10]
selected_campaigns = list(range(1, 11))
rf_param = RF_PARAM_5G.RSRQ

# Data filtering
df = filter_dataframe(
    df=df,
    operators=operator_choice,
    include_columns=[
        "pci",
        "beam_index",
        "nr_arfcn",
        "operator_id",
        "sinr",
        "rsrq"
    ],
    campaigns=selected_campaigns,
)

df["best_beam"] = df["measurements_matrix"].apply(
    lambda x: get_best_beam(x, rf_param)
)


In [ ]:
from scripts.weighted_coverage import wknn_one_tp_row
from scripts.matrix_operations import create_point_matrix, compute_weights
from scripts.utils import dataset_tp_rp_split
from scripts.beamforming import get_beam_sidelobe_pcis
import pandas as pd

# setup
random = 42
use_sidelobes = True


def get_best_beam_diff(matrix: pd.DataFrame, rf_param: RF_PARAM_5G) -> np.float64:
    matrix = matrix.dropna(subset=[rf_param.value])
    idx = matrix.groupby(["pci"])[rf_param.value].idxmax()
    beams = matrix.loc[idx]
    unique_sorted = beams[rf_param.value].sort_values(ascending=False)

    if len(unique_sorted) > 1:
        return unique_sorted[0] - unique_sorted[1]
    return None


def run(random: int):
    # Pre-compute the control matrix (all RPs) once
    df_tp, df_rp = dataset_tp_rp_split(df, 0.3, random)

    # Pre-compute reference point matrices by beam
    rp_matrices_by_beam = {}
    unique_beams = df_rp["best_beam"].unique()

    # Pre-compute matrices for each beam group
    for beam in unique_beams:
        beam_rps = df_rp[df_rp["best_beam"] == beam]

        unique_npcis = [beam] if not use_sidelobes else get_beam_sidelobe_pcis(beam)

        m_rp, idx_rp = create_point_matrix(beam_rps, unique_npcis, rf_param)
        rp_matrices_by_beam[beam] = (m_rp, idx_rp, beam_rps)

    data = []

    for i, (_, tp_row) in enumerate(df_tp.iterrows(), 1):
        tp = pd.DataFrame([tp_row])
        best_beam = tp_row["best_beam"]

        unique_npcis = [best_beam] if not use_sidelobes else get_beam_sidelobe_pcis(best_beam)

        # Get the pre-computed matrices for this beam
        if best_beam not in rp_matrices_by_beam:
            continue

        m_rp, idx_rp, rps = rp_matrices_by_beam[best_beam]

        # Create the point matrix for the test point
        m_tp, idx_tp = create_point_matrix(tp, unique_npcis, rf_param)

        # Compute weights only if we have matching RPs

        W, idx_sort = compute_weights(m_rp, idx_rp, m_tp, idx_tp)
        _, errors = wknn_one_tp_row(tp, rps, idx_sort, W, 2)

        complexity = m_rp.shape[0] * m_rp.shape[1] if len(m_rp.shape) == 2 else None

        # get the difference from the best beam to next best

        mat = tp.iloc[0]['measurements_matrix']
        diff = get_best_beam_diff(mat, rf_param)

        data.append([errors, complexity, diff])

    data = np.array(data)
    return data


data = []
n_runs = 10
for i in range(n_runs):
    print(f'\r{i}/{n_runs}', end='')
    data.extend(run(random_seeds[i]))

results_df = pd.DataFrame(data, columns=["errors", "complexity", "diff"])

results_df

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
plt.scatter(
    results_df["errors"],
    results_df["diff"],
)

plt.grid()
plt.ylim((0, 3))
plt.xlim((0, 50))
plt.xlabel("errors")
plt.ylabel("diff")
plt.show()

In [ ]:
tp = df.iloc[3]

tp

In [ ]:
def get_best_beam_diff(matrix: pd.DataFrame, rf_param: RF_PARAM_5G) -> np.float64:
    matrix = matrix.dropna(subset=[rf_param.value])
    idx = matrix.groupby(["pci"])[rf_param.value].idxmax()
    beams = matrix.loc[idx]
    unique_sorted = beams[rf_param.value].sort_values(ascending=False)

    if len(unique_sorted) > 1:
        return unique_sorted[0] - unique_sorted[1]
    return None


res = get_best_beam_diff(tp['measurements_matrix'], rf_param)

res
